# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

# PENTING: Anda menyertakan DINOv2 secara offline!
MODEL_DINO_DIR = "/kaggle/input/models/metaresearch/dinov2/pytorch/large/1"

# REVISI: KEMBALI KE 384 UNTUK SWIN. 
# DINOv2 akan melakukan penyesuaian dinamis (interpolasi) di Tahap 2.
IMG_SIZE = 384 
BATCH_SIZE = 8
N_FOLDS = 5 

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED K-FOLD
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
df_all['fold'] = -1

for fold, (train_idx, valid_idx) in enumerate(skf.split(X=df_all, y=df_all['label'])):
    df_all.loc[valid_idx, 'fold'] = fold

# ==========================================
# 3. PIPELINE AUGMENTASI KELAS GRANDMASTER
# ==========================================
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    # PILAR 1: Simulasi Lensa & Guncangan Lengan (Motion Blur)
    A.OneOf([
        A.ImageCompression(p=1.0), 
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),                      
        A.MotionBlur(blur_limit=5, p=1.0), 
        A.MedianBlur(blur_limit=5, p=1.0),                   
    ], p=0.4),
    
    # PILAR 2: Simulasi Noise Layar Sentuh & Sensor Kamera Murah
    A.OneOf([
        A.ISONoise(p=1.0),
        A.GaussNoise(p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
    ], p=0.4),
    
    # PILAR 3: Simulasi Pantulan Cahaya & Kontras Lingkungan
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=25, p=1.0),
        A.RandomToneCurve(p=1.0),
    ], p=0.5),
    
    # PILAR 4: Simulasi Kertas Foto Melengkung / Layar Miring (SENJATA RAHASIA)
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=1.0),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-15, 15), p=1.0),
    ], p=0.4),
    
    # Simulasi oklusi (tertutup jari/benda kecil)
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, min_holes=1, fill_value=0, p=0.3),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

print("Prapemrosesan dan Pembagian K-Fold (dengan Augmentasi Ekstrem) Selesai!")
print(f"Total Seluruh Data : {len(df_all)} gambar")

# Membangun Arsitektur Swin 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F  # <--- REVISI: Import fungsi ini untuk interpolasi
import timm
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
import gc

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. FUNGSI PEMBUAT MODEL (FACTORY FUNCTIONS)
# ==========================================
NUM_CLASSES = 6
MODEL_SWIN_NAME = 'swin_base_patch4_window12_384.ms_in22k'

# Direktori Kaggle DINOv2 Large
DINO_DIR = '/kaggle/input/models/metaresearch/dinov2/pytorch/large/1'

# --- PABRIK 1: SWIN TRANSFORMER + ReLU CUSTOM HEAD ---
class SwinWithCustomHead(nn.Module):
    def __init__(self, model_name, num_classes):
        super(SwinWithCustomHead, self).__init__()
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            num_classes=0,       
            drop_rate=0.3,       
            attn_drop_rate=0.2   
        )
        
        num_features = self.backbone.num_features
        
        self.custom_head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Swin Transformer MURNI memakan input 384x384 tanpa diubah
        features = self.backbone(x)
        output = self.custom_head(features)
        return output

def create_swin_model():
    model = SwinWithCustomHead(MODEL_SWIN_NAME, NUM_CLASSES)
    
    total_params = len(list(model.backbone.parameters()))
    freeze_limit = int(total_params * 0.70)
    for i, param in enumerate(model.backbone.parameters()):
        if i < freeze_limit:
            param.requires_grad = False
            
    return model.to(device)


# --- PABRIK 2: DINOv2 LARGE + LoRA (INTERPOLASI DINAMIS) ---
class DinoV2WithCustomHead(nn.Module):
    def __init__(self, model_dir, num_classes):
        super().__init__()
        # Muat dari folder Kaggle
        self.backbone = AutoModel.from_pretrained(model_dir)
        
        num_features = self.backbone.config.hidden_size
        
        self.head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # REVISI PENTING: Interpolasi Dinamis
        # Jika gambar yang masuk BUKAN 392x392 (misalnya 384 dari DataLoader),
        # perbesar gambar tersebut secara otomatis di dalam GPU sebelum masuk ke DINO!
        if x.size(-1) != 392 or x.size(-2) != 392:
            x = F.interpolate(x, size=(392, 392), mode='bilinear', align_corners=False)
            
        outputs = self.backbone(pixel_values=x)
        cls_features = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_features)

def create_dino_lora_model():
    print(f"Memuat DINOv2 Large secara LURING dari direktori Kaggle...")
    
    base_model = DinoV2WithCustomHead(DINO_DIR, NUM_CLASSES)
    
    lora_config = LoraConfig(
        r=16,                  
        lora_alpha=16,         
        target_modules=["query", "key", "value"], 
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["head"] 
    )
    
    lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters() 
    
    return lora_model.to(device)


# ==========================================
# 3. UJI COBA FUNGSI (SANITY CHECK)
# ==========================================
print("\nMenguji coba pembuatan kedua model elit...")
temp_swin = create_swin_model()
temp_dino_lora = create_dino_lora_model()

# REVISI: SEKARANG KITA UJI MENGGUNAKAN INPUT 384x384 !
# Jika berhasil lolos tanpa error, berarti interpolasi DINO bekerja sempurna.
dummy_input = torch.randn(2, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_out_swin = temp_swin(dummy_input)
    dummy_out_dino = temp_dino_lora(dummy_input)

print("\nFungsi pembuat model berhasil diverifikasi dan siap mencetak 10 model!")
print(f"Bentuk input (Sesuai DataLoader) : {dummy_input.shape}")
print(f"Bentuk output Swin               : {dummy_out_swin.shape}")
print(f"Bentuk output DINO (Interpolasi) : {dummy_out_dino.shape}")

del temp_swin, temp_dino_lora
torch.cuda.empty_cache()
gc.collect()

# Strategi Pelatihan (Warm-up & Loss)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np
import gc

# ==========================================
# 1. DEFINISI FOCAL LOSS (LABEL SMOOTHING SAJA)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean', label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        smooth_target = F.one_hot(targets, num_classes=num_classes).float()
        smooth_target = smooth_target * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        
        log_pt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(log_pt)
        
        ce_loss = -(smooth_target * log_pt).sum(dim=1)
        F_loss = self.alpha * (1 - pt.gather(1, targets.view(-1, 1)).squeeze(1))**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# ==========================================
# 2. FUNGSI PELATIHAN GENERIC (MURNI)
# ==========================================
def train_fold(fold, train_loader, valid_loader, model_func, model_prefix, epochs=30, lr=1e-5, patience=10, accumulation_steps=4):
    print(f"\n{'='*50}")
    print(f"Memulai Pelatihan {model_prefix.upper()}: FOLD {fold}")
    print(f"{'='*50}")
    
    model = model_func()
    criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = torch.amp.GradScaler('cuda')

    best_val_f1 = 0.0
    patience_counter = 0
    save_path = f"best_{model_prefix}_fold_{fold}.pth"

    for epoch in range(epochs):
        # --- FASE TRAINING ---
        model.train()
        train_loss = 0.0
        optimizer.zero_grad() 
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train {model_prefix} Fold {fold}]")
        for batch_idx, (images, labels) in enumerate(train_bar):
            images, labels = images.to(device), labels.to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images) 
                loss = criterion(outputs, labels)
                loss = loss / accumulation_steps 
            
            scaler.scale(loss).backward() 
            
            if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(train_loader)):
                scaler.step(optimizer) 
                scaler.update()
                optimizer.zero_grad() 
            
            train_loss += loss.item() * accumulation_steps * images.size(0)
            train_bar.set_postfix(loss=(loss.item() * accumulation_steps))
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # --- FASE VALIDASI ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid {model_prefix} Fold {fold}]")
        with torch.no_grad(): 
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nHasil Epoch {epoch+1} ({model_prefix} Fold {fold}):")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        
        scheduler.step()
        
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            print(f"Model membaik! Disimpan ke '{save_path}'")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"Model tidak membaik. Kesabaran: {patience_counter}/{patience}")
            
        if patience_counter >= patience:
            print(f"\nEarly Stopping dipicu untuk {model_prefix} Fold {fold}!")
            break

    print(f"\nPelatihan {model_prefix} Fold {fold} Selesai! Skor Macro F1 Terbaik: {best_val_f1:.4f}")
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_val_f1

# ==========================================
# 3. EKSEKUSI PELATIHAN 10 MODEL SUPER
# ==========================================
swin_fold_scores = []
dino_fold_scores = []

for fold in range(N_FOLDS):
    train_df_fold = df_all[df_all['fold'] != fold].reset_index(drop=True)
    valid_df_fold = df_all[df_all['fold'] == fold].reset_index(drop=True)
    
    train_dataset_fold = SpoofingDataset(train_df_fold, transforms=train_transforms)
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- MELATIH SWIN TRANSFORMER ---
    best_swin_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_swin_model, 
        model_prefix="swin",
        lr=1e-5
    )
    swin_fold_scores.append(best_swin_f1)
    
    # --- MELATIH DINOv2 LARGE + LoRA ---
    best_dino_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_dino_lora_model,
        model_prefix="dino",               
        lr=3e-5 
    )
    dino_fold_scores.append(best_dino_f1)

# ==========================================
# 4. REKAPITULASI AKHIR
# ==========================================
print(f"\n{'='*50}")
print("REKAPITULASI PELATIHAN 10-MODEL ENSEMBLE (MURNI)")
print(f"{'='*50}")

print("--- SWIN TRANSFORMER ---")
for i, score in enumerate(swin_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata Swin CV F1-Score : {np.mean(swin_fold_scores):.4f}\n")

print("--- DINOv2 LARGE + LoRA ---")
for i, score in enumerate(dino_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata DINO CV F1-Score  : {np.mean(dino_fold_scores):.4f}")

# Kalibrasi Probabilitas (Thresholding)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import DataLoader
import warnings
import gc

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6
N_FOLDS = 5 

# Menyiapkan array kosong untuk menampung probabilitas (Swin, DINOv2, dan Gabungan)
oof_probs_swin = np.zeros((len(df_all), NUM_CLASSES))
oof_probs_dino = np.zeros((len(df_all), NUM_CLASSES))
oof_labels = np.zeros(len(df_all))

# ==========================================
# 1. EKSTRAKSI PROBABILITAS OUT-OF-FOLD (OOF)
# ==========================================
print("=== EKSTRAKSI PROBABILITAS OUT-OF-FOLD (SWIN + DINOv2) ===")

for fold in range(N_FOLDS):
    print(f"\nMengekstrak prediksi dari Fold {fold}...")
    
    val_idx = df_all[df_all['fold'] == fold].index
    valid_df_fold = df_all.iloc[val_idx].reset_index(drop=True)
    
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- A. EKSTRAKSI SWIN TRANSFORMER ---
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()
    
    fold_probs_swin = []
    fold_labels = []
    
    with torch.no_grad():
        for images, labels in valid_loader_fold: 
            images = images.to(device)
            out_swin = model_swin(images)
            probs_swin = F.softmax(out_swin, dim=1) 
            fold_probs_swin.extend(probs_swin.cpu().numpy())
            fold_labels.extend(labels.numpy())
            
    oof_probs_swin[val_idx] = np.array(fold_probs_swin)
    oof_labels[val_idx] = np.array(fold_labels)
    
    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

    # --- B. EKSTRAKSI DINOv2 LARGE + LoRA ---
    model_dino = create_dino_lora_model()
    model_dino.load_state_dict(torch.load(f'best_dino_fold_{fold}.pth', map_location=device))
    model_dino.eval()
    
    fold_probs_dino = []
    
    with torch.no_grad():
        for images, _ in valid_loader_fold: 
            images = images.to(device)
            out_dino = model_dino(images)
            probs_dino = F.softmax(out_dino, dim=1) 
            fold_probs_dino.extend(probs_dino.cpu().numpy())
            
    oof_probs_dino[val_idx] = np.array(fold_probs_dino)
    
    del model_dino
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 2. PENGATURAN RASIO ENSEMBLE MANUAL (LB OPTIMIZED)
# ==========================================
print("\n--- MENGGUNAKAN RASIO ENSEMBLE MANUAL ---")
# REVISI: Mengembalikan dominasi ke DINOv2 (75%) karena terbukti lebih kebal 
# terhadap data Test Kaggle yang tersembunyi. Swin Transformer mendapat 25%.
best_alpha = 0.25

print(f"Rasio Ditetapkan : {best_alpha:.2f} Swin : {1.0 - best_alpha:.2f} DINOv2")

# ==========================================
# 3. EVALUASI AKHIR (MURNI TANPA OFFSETS)
# ==========================================
# Terapkan rasio manual ke probabilitas gabungan
oof_probs_ensemble = (oof_probs_swin * best_alpha) + (oof_probs_dino * (1.0 - best_alpha))
preds_after = np.argmax(oof_probs_ensemble, axis=1)

blend_f1 = f1_score(oof_labels, preds_after, average='macro')
print(f"Macro F1-Score Gabungan (Lokal) : {blend_f1:.5f}")

print("\n--- EVALUASI AKHIR ---")
print(classification_report(oof_labels, preds_after, target_names=class_names, digits=4))

# Prediksi TTA & Pengumpulan (Submission)

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
import gc

warnings.filterwarnings("ignore")

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

IMG_SIZE = 384 
BATCH_SIZE = 8 
N_FOLDS = 5

test_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6

# ==========================================
# 3. PENGATURAN BLENDING MANUAL (LB OPTIMIZED)
# ==========================================
# REVISI: Mengunci rasio 75% untuk DINOv2 dan 25% untuk Swin Transformer
best_alpha = 0.25 

print("\n--- PARAMETER PENGGABUNGAN (BLENDING) MANUAL ---")
print(f"Rasio Swin Transformer : {best_alpha:.2f}")
print(f"Rasio DINOv2 Large     : {1.0 - best_alpha:.2f}")

# ==========================================
# 4. PREDIKSI 10-MODEL ENSEMBLE DENGAN TTA
# ==========================================
print(f"\nMemulai prediksi {len(sub_df)} data dengan TTA (10-Model Ensemble Mode)...")

final_swin_probs = np.zeros((len(sub_df), NUM_CLASSES))
final_dino_probs = np.zeros((len(sub_df), NUM_CLASSES)) 

# --- BAGIAN A: PREDIKSI 5 SWIN TRANSFORMER ---
print("\n=== EKSEKUSI 5 MODEL SWIN TRANSFORMER ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan Swin Fold {fold}...")
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()

    fold_preds_swin = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"Swin TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            prob_orig = F.softmax(model_swin(imgs_orig), dim=1)
            prob_flip = F.softmax(model_swin(imgs_flip), dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_swin.extend(prob_avg.cpu().numpy())

    final_swin_probs += (np.array(fold_preds_swin) / N_FOLDS)

    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

# --- BAGIAN B: PREDIKSI 5 DINOv2 LARGE + LoRA ---
print("\n=== EKSEKUSI 5 MODEL DINOv2 LARGE + LoRA ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan DINO Fold {fold}...")
    model_dino = create_dino_lora_model() 
    model_dino.load_state_dict(torch.load(f'best_dino_fold_{fold}.pth', map_location=device))
    model_dino.eval()

    fold_preds_dino = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"DINO TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            prob_orig = F.softmax(model_dino(imgs_orig), dim=1)
            prob_flip = F.softmax(model_dino(imgs_flip), dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_dino.extend(prob_avg.cpu().numpy())

    final_dino_probs += (np.array(fold_preds_dino) / N_FOLDS)

    del model_dino
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 5. PENERAPAN MURNI & PENYIMPANAN
# ==========================================
print(f"\n--- Menggabungkan 10 Prediksi (Rasio Swin {best_alpha:.2f}) ---")

# Gabungkan menggunakan rasio manual yang ditetapkan di awal
final_ensemble_probs = (final_swin_probs * best_alpha) + (final_dino_probs * (1.0 - best_alpha))
final_preds = np.argmax(final_ensemble_probs, axis=1)

# Simpan ke CSV
final_labels = [id2label[pred] for pred in final_preds]
sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' Super 10-Model (LB Optimized) berhasil dibuat.")

In [ ]:
# DINO 